In [36]:
from f_0_dirs import get_data_dirs
dirs = get_data_dirs()

# Iterate over the attributes of the DirPaths object
for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue
    print(f"{attr}: {getattr(dirs, attr)}")

data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.07.30
output_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\output
raw_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.02
root_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean
root_dir: C:\Users\lazyst\Files\ucl\Dissertation
work_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\src


# 1. [read] from FAME

### Categorise raw files

This script resolves the project data paths, scans the raw-data folder, builds a nested dictionary of file metadata by company and file category, and writes it to JSON.  
`get_data_dirs()` defines the working directories  
`build_raw_file_dict()` performs the recursive traversal and file collection through its helper functions.  

In [37]:
from flask import json
import pandas as pd

def convert(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60
    return "%dh:%02dm:%02ds" % (hour, minutes, seconds)
rate = 1 # 1 second per file

from f_1_traverse import build_raw_file_dict
pd.options.mode.chained_assignment = None  # default='warn'

if dirs.raw_data_dir is None:
	raise ValueError("raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

raw_file_dict, file_count, processed = build_raw_file_dict(dirs.raw_data_dir)
with open(dirs.output_dir / "raw_file_dict.json", "w") as f:
    json.dump(raw_file_dict, f, indent=4)
    print(f"✅ Successfully built raw file dictionary and saved to: {dirs.output_dir / 'raw_file_dict.json'}")
    print(f"✅ Processed {processed} file paths.")

# Count files I'll need to process
file_count_df = pd.DataFrame(file_count).T.reset_index().rename(columns={'index': 'industry'})
file_count_df['total'] = file_count_df.drop(columns=['industry']).sum(axis=1)
with open(dirs.root_dir / "build" / "tmp" / "file_count.md", "w") as f:
    f.write(file_count_df.to_markdown())
    print(f"✅ Successfully built file count markdown table and saved to: {dirs.root_dir / 'build' / 'tmp' / 'file_count.md'}")
# Get highest total files and the corresponding industry, and highest individual folder that will need to be processed
ind_max_row = file_count_df.loc[file_count_df['total'].idxmax()]
ind_max_id = ind_max_row['industry']
ind_max = ind_max_row['total']

# Unpivot the dataframe with columns 'industry', 'property', and 'value' where property will be one of a1_ID, a2_key_finance up to a5.
file_count_melted = file_count_df.melt(id_vars=['industry'], var_name='property', value_name='value')
# Drop rows where property is 'total'
file_count_melted = file_count_melted[file_count_melted['property'] != 'total']
prop_max_row = file_count_melted.loc[file_count_melted['value'].idxmax()]
prop_max_ind = prop_max_row['industry']
prop_max_prop = prop_max_row['property']
prop_max = prop_max_row['value']
# Assume 80 MB of RAM is required per file (dataframe)
peak_ram_required = prop_max * 6 / 1024 * 2 # in GB

print(f"⚠️ Max industry files: industry '{ind_max_id}' with {ind_max} files.")
print(f"⚠️ Max property files: industry '{prop_max_ind}'/property '{prop_max_prop}' with {prop_max} files.")
print(f"⚠️ Estimated peak RAM required: {peak_ram_required:.2f} GB (for industry '{prop_max_ind}' and property '{prop_max_prop}')")
print(f"⏱️ Estimated time to process all files: {convert(processed * rate)} (at {rate} second per file)")

Traversing industry directory: 01, 02, 03, 05, 06, 07, 08, 09, 10, 11, 12, 13, 14, 15, 16
17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31
32, 33, 35, 36, 37, 38, 39, 41, 42, 43, 45, 46, 47, 49, 50
51, 52, 53, 55, 56, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68
69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 82, 84, 85
86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99
✅ Successfully built raw file dictionary and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\output\raw_file_dict.json
✅ Processed 12846 file paths.
✅ Successfully built file count markdown table and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\file_count.md
⚠️ Max industry files: industry '47' with 1090 files.
⚠️ Max property files: industry '47'/property 'a4_profits' with 431 files.
⚠️ Estimated peak RAM required: 5.05 GB (for industry '47' and property 'a4_profits')
⏱️ Estimated time to process all files: 3h:34m:06s (at 1 second per file)


Load in the master schema

In [38]:
from typing import TypedDict
import pandas as pd

# import xlsx file from input/raw_properties.xlsx to load as a schema
# Declare types that the schema_source df has colums ["from_raw", "key", "type", "fuzzy_mapping", "in_raw_data", "keep", "in_ln_set", "description"]
schema_path = dirs.root_dir / "build" / "input" / "raw_properties.xlsx"

# cast types
# columns 'in_ln_set' and 'may_mix' are boolean, but some rows are nan
# cast nan to false and ensure the column is a proper boolean
schema_source = pd.read_excel(schema_path, sheet_name="raw_properties", engine="calamine",
    dtype={
        "from_raw": str,
        "key": str,
        "type": str,
        "fuzzy_mapping": str,
        "keep": str,
        "in_ln_set": "boolean",
        "may_mix": "boolean",
        "description": str
    }
)
# Iterate over columns where type is boolean and fillna with False
for col in schema_source.select_dtypes(include='boolean').columns:
    schema_source[col] = schema_source[col].fillna(False)
schema_raw: pd.DataFrame = schema_source[schema_source["from_raw"].notna()]

# Take our master schema and turn it into a helpful mapping of fuzzy column names to schema column names
# Regardless of what the raw source is. We'll filter it later
class SRFbyProperty(TypedDict):
    mapping: pd.DataFrame
    col_map: dict[str, str]

properties = ['a1_ID', 'a2_key_finance', 'a3_assets', 'a4_profits', 'a5_misc']
schema_fixed_fuzzy_df: pd.DataFrame = schema_source[["key", "fuzzy_mapping"]]
schema_fixed_fuzzy_by_property = {}
for p in properties:
    srf_filtered: pd.DataFrame = schema_raw[schema_raw["from_raw"].isin([p, 'all'])]
    srf_filtered["fml"] = srf_filtered["fuzzy_mapping"].str.split('\n')
    # Just make 
    srf_mapping: pd.DataFrame = srf_filtered
    # For every value in srf_filtered["fml"]
    # Create a dict which is that value, and the corresponding entry in srf_filtered["key"]
    # But only if the value is not NaN
    srf_col_map = {
        fml: key
        for _, row in srf_filtered.iterrows()
        if row["fml"] is not None and isinstance(row["fml"], list)
        for fml in row["fml"]
        for key in [row["key"]]
    }
    schema_fixed_fuzzy_by_property[p] = {
        "mapping": srf_mapping,
        "col_map": srf_col_map
    }

# Print any rows where fml has more than one element
for p in properties:
    mapping = schema_fixed_fuzzy_by_property[p]["mapping"]
    for index, row in mapping.iterrows():
        if row["fml"] is None:
            print(f"Row {index} has fml that is None: {row['key']}")
        elif type(row["fml"]) is not list:
            print(f"Row {index} has fml that is not a list: {row['key']}")
        elif len(row["fml"]) > 1:
            print(f"Row {index} has more than one fuzzy mapping: {row['fml']}")

Row 37 has more than one fuzzy mapping: ['Strategy,  organization and policy', 'Strategy, organization and policy']
Row 53 has fml that is not a list: industry_codes
Row 54 has fml that is not a list: file_codes
Row 53 has fml that is not a list: industry_codes
Row 54 has fml that is not a list: file_codes
Row 53 has fml that is not a list: industry_codes
Row 54 has fml that is not a list: file_codes
Row 53 has fml that is not a list: industry_codes
Row 54 has fml that is not a list: file_codes
Row 53 has fml that is not a list: industry_codes
Row 54 has fml that is not a list: file_codes


# 2. [write] To duck schemas

### Define duck schemas

In [39]:
import pandas as pd
# Import ibis-framework
import ibis
# pip install 'ibis-framework[duckdb,geospatial]'

print("Path:", ibis.__file__)
print("Version:", getattr(ibis, "__version__", "No version found"))
pd.options.mode.chained_assignment = None  # default='warn'

db_path = dirs.output_dir / "fame_data.duckdb"

# Fixed schema
# schema_fixed is a df of schema_source where values in column "keep" are "fixed" or "all"
schema_fixed: pd.DataFrame = schema_source[schema_source["keep"].isin(["fixed", "all"])]
schema_fixed_dict: dict[str, str] = dict(zip(schema_fixed["key"], schema_fixed["type"]))
schema_fixed_ibis: ibis.Schema = ibis.schema(schema_fixed_dict)
schema_fixed_names: set[str] = set(schema_fixed_ibis.keys())

schema_derived: pd.DataFrame = schema_source[schema_source["keep"].isin(["derived", "all"])]
schema_derived_dict: dict[str, str] = dict(zip(schema_derived["key"], schema_derived["type"]))
schema_derived_ibis: ibis.Schema = ibis.schema(schema_derived_dict)
schema_derived_names: set[str] = set(schema_derived_ibis.keys())

# build a schema_yearly df with columns registered_number, fame_key, year, value properties, with types str, str, int, float
schema_yearly: pd.DataFrame = pd.DataFrame({
    "key": ["registered_number", "fame_key", "year", "value"],
    "type": ["string", "string", "int64", "float64"]
})
schema_yearly_dict: dict[str, str] = dict(zip(schema_yearly["key"], schema_yearly["type"]))
schema_yearly_ibis: ibis.Schema = ibis.schema(schema_yearly_dict)
schema_yearly_names: set[str] = set(schema_source[
    (schema_source["keep"] == "yearly") &
    (schema_source["from_raw"].notna())]["key"]
)

# 4. Execute the table creation using the Ibis schema
try:
    
    # 2. Connect to DuckDB using Ibis
    con = ibis.duckdb.connect(str(db_path))
    print(f"Initializing DuckDB via Ibis at: {db_path}")
    # overwrite=True prevents errors if the script is run multiple times during setup
    con.create_table("fame_fixed", schema=schema_fixed_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_fixed'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_fixed").schema())

    con.create_table("fame_derived", schema=schema_derived_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_derived'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_derived").schema())

    con.create_table("fame_yearly", schema=schema_yearly_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_yearly'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_yearly").schema())

except Exception as e:
    print(f"❌ Error creating table: {e}")

Path: c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\__init__.py
Version: 12.0.0
Initializing DuckDB via Ibis at: C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb
✅ Successfully created Ibis schema for 'fame_fixed'.

Table Schema Verification:
ibis.Schema {
  company_name                       string
  registered_number                  string
  ticker_symbol                      string
  primary_trading_address            string
  primary_trading_address_latitude   string
  primary_trading_address_longitude  string
  branch_name                        string
  primary_uk_sic_2007_code           int64
  primary_uk_sic_2007_description    string
  latest_accounts_date               date
  no_of_available_years              int64
  guo                                string
  guo_nb                             int64
  entity_type                        string
}
✅ Successfully created Ibis schema for 'fame_derived'.

Table Schema Verifica

### Load, modify and write imported file schema

In [40]:
import traceback

from flask import json
from f_1_traverse import RawFileDict
from f_2_check import drop_duplicate_columns, check_df_matches_schema, handle_excel_dates, handle_mixed_types, rename_df_with_years
from f_2_modify import coerce_ibis_dates_from_schema, reindex_ibis_table
import random

shuffle_flag = False

# Traverse the raw_file_dict.json file to get each Excel filepath
# declare raw_file_dict as a RawFileDict type
raw_file_dict: RawFileDict | None = None
with open(dirs.output_dir / "raw_file_dict.json", "r") as f:
    raw_file_dict = json.load(f)
if raw_file_dict is None:
    raise ValueError("❌ Error: raw_file_dict.json is empty or not found.")
if dirs.raw_data_dir is None:
    raise ValueError("❌ Error: raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

# Handle yearly variables
start_year = 2006
end_year = 2025

# We want one big dataframe, which we will merge all the data into for now
# df_fixed = pd.DataFrame(columns=list(schema_fixed.columns))
ind_keys = raw_file_dict.keys()
ind_shuffled = list(ind_keys)
random.shuffle(ind_shuffled)

process_count = 0
for ind in ind_shuffled if shuffle_flag else ind_keys:
    obj = raw_file_dict[ind]

    print(f"Ingesting industry: {ind} with {len(obj)} properties.")
    process_count += 1
    if process_count > 2:
        break

    batches_raw: dict[str, list[pd.DataFrame]] = { p: [] for p in properties }
    ind_batches_masters: dict[str, pd.DataFrame] = { p: pd.DataFrame() for p in properties }

    for property, arr in obj.items():
        if not arr:
            print(f"⚠️ Warning: No files found for property '{property}' in industry '{ind}'. Skipping.")
            continue

        print(f"--- Ingesting property: {property} with {len(arr)} files.")
        property_batch: list[pd.DataFrame] = []
        schema_raw_fuzzy_col_map = schema_fixed_fuzzy_by_property[property]['col_map']

        files_shuffled = arr.copy()
        random.shuffle(files_shuffled)
        for [file_name, file_path] in files_shuffled:

            # LOAD: df_raw has no fixed schema so we can ingest and modify it as pleases
            df_raw = pd.read_excel(file_path, engine='calamine', sheet_name='Results', header=0, dtype={
                "Registered number": str                                            # "Leading Zeros" Trap. Pandas accidentally processes
            })                                                                      #   registered number as int64 when reading the file, which will chop zeros.
            df_raw.drop(df_raw.columns[0], axis=1, inplace=True)                    # Drop column A (blank in raw data)
            df_raw = drop_duplicate_columns(df_raw)                                 # Remove duplicate columns (quirk of some files)

            df_raw = rename_df_with_years(
                df_raw, schema_raw_fuzzy_col_map, property, start_year, end_year,
                ref=f"{ind}/{property}/{file_name}"
            )
            df_raw['registered_number'] = df_raw['registered_number'].astype(str).str.replace(r'\.0$', '', regex=True)      # Ensures they are strings
            
            # # Drop columns where the corresponding 'from_raw' entry in the schema_raw
            # # Doesn't match the current property or 'all'. This ensures we only keep relevant columns for the current property.
            # p_cols = schema_raw[schema_raw["from_raw"].isin([property, 'all'])]["key"].tolist()
            # df_raw = df_raw[p_cols]

            # Drop columns where the corresponding 'from_raw' entry in the schema_raw
            # Doesn't match the current property or 'all'.
            # Unless the 'keep' column is set to yearly
            # In which case the column name only has to start with the 'key' for that property
            # This ensures we only keep relevant columns for the current property.
            exact_cols = schema_raw[(schema_raw["from_raw"].isin([property, 'all'])) & (schema_raw["keep"] != "yearly")]["key"].tolist()
            yearly_cols = schema_raw[(schema_raw["from_raw"].isin([property, 'all'])) & (schema_raw["keep"] == "yearly")]["key"].tolist()
            df_raw_cols = [col for col in df_raw.columns if col in exact_cols or any(col.startswith(yc) for yc in yearly_cols)]
            df_raw = df_raw[df_raw_cols]

            df_raw = df_raw[df_raw['registered_number'].notna()]      
            df_raw = handle_mixed_types(schema_raw, df_raw, ref=f"{ind}/{property}/{file_name}")
            df_raw = handle_excel_dates(df_raw)                                     # Handle any Excel date serials to datetime
            df_raw2 = df_raw.copy()                                                 # Defragment
            df_raw2['industry_codes'] = ind
            df_raw2['file_codes'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
            property_batch.append(df_raw2)
        ind_batches_masters[property] = pd.concat(
            property_batch, ignore_index=True
        ).drop_duplicates(
            subset=['registered_number'], keep='first'
        ).copy()
        del property_batch

    print(f"Deriving ingested files for industry: {ind}.")
    
    try:
        for p in properties:
            if p in ind_batches_masters:
                continue
            raise ValueError(f"❌ Error: No data found for property '{p}' in industry '{ind}'. Please check the raw data files.")

        # FILTER, processing the a1_ID table first.
        schema_raw_fuzzy_mapping = schema_fixed_fuzzy_by_property["a1_ID"]["mapping"]
        df_id = ind_batches_masters["a1_ID"]
        if 'no_of_available_years' in df_id.columns:
            df_id = df_id[df_id['no_of_available_years'] != 0]
        if 'ro_country' in df_id.columns:
            df_id = df_id[df_id['ro_country'] != "Republic of Ireland"]
        ind_batches_masters["a1_ID"] = df_id

        try:
            check_df_matches_schema(schema_raw_fuzzy_mapping, ind_batches_masters["a1_ID"])           # Check if the DataFrame matches the input
        except ValueError as e:
            print(f"⚠️ Mismatch in raw data validation for file {ind}/a1_ID)")
            print("Warning:", e)

        # MODIFY
        table_t_id = ibis.memtable(ind_batches_masters["a1_ID"])
        table_t_misc = ibis.memtable(ind_batches_masters["a5_misc"])
        table_t_merged_id_misc = table_t_id.left_join(table_t_misc, "registered_number").select(
            *[table_t_id[col] for col in table_t_id.columns],
            *[table_t_misc[col] for col in table_t_misc.columns if col not in table_t_id.columns]
        )
        table_t_fixed = reindex_ibis_table(schema_fixed_ibis, table_t_merged_id_misc)
        table_t_fixed_dates = coerce_ibis_dates_from_schema(schema_fixed_ibis, table_t_fixed)
        table_fixed_type_casts = {
            col: table_t_fixed_dates[col].try_cast(schema_fixed_ibis.fields[col])
            for col in schema_fixed_names if col in table_t_fixed_dates.columns
        }
        table_t_fixed_cast = table_t_fixed_dates.mutate(**table_fixed_type_casts).select(schema_fixed_names)
        con.insert("fame_fixed", table_t_fixed_cast)
        print(f"✅ Successfully appended fixed table from: {ind}")
            
        # Safely extract columns, defaulting to an Ibis null object if missing from raw data
        col_pta = table_t_id['primary_trading_address'] if 'primary_trading_address' in table_t_id.columns else ibis.null()
        col_lat = table_t_id['primary_trading_address_latitude'] if 'primary_trading_address_latitude' in table_t_id.columns else ibis.null()
        col_lon = table_t_id['primary_trading_address_longitude'] if 'primary_trading_address_longitude' in table_t_id.columns else ibis.null()
        col_ticker = table_t_id['ticker_symbol'] if 'ticker_symbol' in table_t_id.columns else ibis.null()
        col_comp = table_t_id['company_name'] if 'company_name' in table_t_id.columns else ibis.null()
        col_branch = table_t_id['branch_name'] if 'branch_name' in table_t_id.columns else ibis.null()
        col_ind = table_t_id['industry_codes'] if 'industry_codes' in table_t_id.columns else ibis.null()
        col_file = table_t_id['file_codes'] if 'file_codes' in table_t_id.columns else ibis.null()

        table_derived = table_t_id.mutate(
            has_ptaddress = col_pta.notnull(),
            has_ptaddress_latlong = col_lat.notnull() & col_lon.notnull(),
            is_public = col_ticker.notnull(),
            has_company_branch_mismatch = col_comp != col_branch,
            industry_codes = col_ind,
            file_codes = col_file
        ).select(schema_derived_names)

        con.insert("fame_derived", table_derived)
        print(f"✅ Successfully appended derived table from: {ind}")

        for p in ["a2_key_finance", "a3_assets", "a4_profits"]:
            pass

            # TODO: for yearly variables of the form 'my_key 2005', we need to extract the year and somehow store it
            # TODO: then fuzzy match the remainder of the column name to the schema like all the other tables
            # TODO: saving that year for later so we can use it in an unzipped format for the yearly table

    except Exception as e:
        print(f"❌ Error processing folder {ind}")
        print(f"❌ Pipeline failed: {type(e).__name__} - {e}")
        traceback.print_exc() # This prints the full red error log so you know the exact line
    

# Drop all temporary tables
for table_name in con.list_tables():
    # continue if the table name is fame_fixed or fame_derived
    if table_name in ["fame_fixed", "fame_derived", "fame_yearly"]:
        continue
    try:
        con.drop_table(table_name)
        print(f"✅ Successfully dropped table {table_name}.")
    except:
        pass

Ingesting industry: 01 with 5 properties.
--- Ingesting property: a1_ID with 5 files.
⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
--- Ingesting property: a2_key_finance with 10 files.
--- Ingesting property: a3_assets with 32 files.
--- Ingesting property: a4_profits with 38 files.
--- Ingesting property: a5_misc with 3 files.
Deriving ingested files for industry: 01.
✅ Successfully appended fixed table from: 01
✅ Successfully appended derived table from: 01
Ingesting industry: 02 with 5 properties.
--- Ingesting property: a1_ID with 1 files.
--- Ingesting property: a2_key_finance with 2 files.
--- Ingesting property: a3_assets with 5 files.
--- Ingesting property: a4_profits with 6 files.
--- Ingesting property: a5_misc with 1 files.
Deriving ingested files for industry: 02.
✅ Successfully appended fixed table from: 02
✅ Successfully appended derived table from: 02
Ingesting industry: 03 with 5 properties.


# 3. [view] resulting DB for inspection

### basic tables overview

In [44]:
# List tables in the DuckDB database as an .md file in /tmp
# Give me the head of all tables
# Ensure they are nicely formatted with headers so I can easily see what's going on
out_file = dirs.root_dir / "build" / "tmp" / "duckdb_tables.md"
with open(out_file, "w") as f:
    tables = con.list_tables()
    f.write("# Tables in DuckDB database\n\n")
    for table in tables:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute()}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).execute().head()}\n```\n\n")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

✅ Successfully listed tables and their heads in: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\duckdb_tables.md


### Check for duplicates in the resulting tables
Check for any duplicate registered_number values in fame_fixed and fame_derived
Print a json in dirs.root_dir / "build" / "tmp" which is an array of objects, which contains properties:
- registered_number
- company_name
- industry_codes
- file_codes
- rows: the number of instances of this registered_number in this table
- all_other_properties_identical: a boolean indicating whether all other properties are identical across instances
- differing_properties: if and only iff all_other_properties_identical is False,
this will be a dict of the differing properties and their values across instances
In general within the for loop, use if not and then continue statements, rather than increasingly nested indentation

In [42]:
import json
import pandas as pd

for table_name in ["fame_fixed", "fame_derived"]:
    
    output_duplicates = []
    output_duplicate_paths = dirs.root_dir / "build" / "tmp" / f"duplicates_{table_name}.json"
    table = con.table(table_name)
    
    # 1. Bulk Database Operation: Identify duplicates in DuckDB
    dup_counts = (
        table.group_by("registered_number")
        .aggregate(rows=table.registered_number.count())
        .filter(ibis._.rows > 1)
    )
    
    # Check if any duplicates exist without pulling all data
    if dup_counts.count().execute() == 0:
        print(f"✅ No duplicate registered_number values found in {table_name}.")
        continue

    # 2. The N+1 Fix: Fetch ALL duplicate rows in exactly ONE query using an inner join
    joined_table = table.inner_join(dup_counts, "registered_number")
    all_dup_instances = joined_table.execute()  # Single execution brings everything into memory

    # 3. Process in Memory: Use Pandas groupby for lightning-fast aggregation
    for registered_number, group in all_dup_instances.groupby("registered_number"):
        
        # The 'rows' column was added by our aggregation join above
        rows = int(group["rows"].iloc[0])
        
        # Helper to safely extract single values and convert Pandas NaN to None
        def get_first(col):
            val = group[col].iloc[0] if col in group.columns else None
            return None if pd.isna(val) else val
            
        company_name = get_first("company_name")
        industry_codes = get_first("industry_codes")
        file_codes = get_first("file_codes")
        
        differing_properties = {}
        
        # Get columns to check (ignore primary key and the joined 'rows' count)
        check_cols = [c for c in group.columns if c not in ["registered_number", "rows"]]
        
        # dropna=False ensures we catch differences between a valid value and a missing value (NaN)
        nunique_counts = group[check_cols].nunique(dropna=False)
        
        # Filter down to columns that have more than 1 unique value
        diff_cols = nunique_counts[nunique_counts > 1].index.tolist()
        
        for col in diff_cols:
            # Extract unique values, replace Pandas NaN/NaT with standard Python None for JSON serialization
            # Ignore industry_codes and file_codes since they are expected to differ across duplicates
            if col in ["industry_codes", "file_codes"]:
                continue
            unique_vals = group[col].unique()
            clean_vals = [None if pd.isna(x) else str(x) if isinstance(x, pd.Timestamp) else x for x in unique_vals]
            differing_properties[col] = clean_vals
            
        all_other_properties_identical = len(differing_properties) == 0
        
        output_duplicates.append({
            "table": table_name,
            "registered_number": str(registered_number),
            "company_name": str(company_name) if company_name else None,
            "industry_codes": str(industry_codes) if industry_codes else None,
            "file_codes": str(file_codes) if file_codes else None,
            "rows": rows,
            "all_other_properties_identical": all_other_properties_identical,
            "differing_properties": differing_properties
        })

    # Tell me how many entries there are which don't have properties identical
    # Print those instances below in this terminal
    non_identical_duplicates = [dup for dup in output_duplicates if not dup["all_other_properties_identical"]]
    print(f"❌ Found {len(non_identical_duplicates)} duplicate entries in {table_name} with non-identical properties.")
    for dup in non_identical_duplicates:
        print(f"  - {dup['registered_number']}: {dup['differing_properties']}")
        # Put these entries first in the output_duplicates list so they are easier to find in the JSON file
        output_duplicates.remove(dup)
        output_duplicates.insert(0, dup)

    with open(output_duplicate_paths, "w") as f:
        json.dump(output_duplicates, f, indent=4, default=str)
        print(f"✅ Successfully checked for duplicate registered_number values and saved to: {output_duplicate_paths}")

❌ Found 0 duplicate entries in fame_fixed with non-identical properties.
✅ Successfully checked for duplicate registered_number values and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\duplicates_fame_fixed.json
❌ Found 0 duplicate entries in fame_derived with non-identical properties.
✅ Successfully checked for duplicate registered_number values and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\duplicates_fame_derived.json


### post-hoc derived table merging on industry_codes

This script natively groups the boolean flags, isolates the unique string codes, concatenates them with commas, joins them back together, and overwrites the table with the pristine data.

In [ ]:
import ibis

t = con.table("fame_derived")
t_count = t.count().execute()

# 1. Inspect the table dynamically (No hardcoding schema names!)
concat_cols = ["industry_codes", "file_codes"]
standard_cols = [c for c in t.columns if c not in concat_cols and c != "registered_number"]

# 2. Tell DuckDB how to merge the standard identical columns.
# Taking .max() safely grabs the identical value across the duplicate rows.
aggs = {c: t[c].max() for c in standard_cols}
clean_table = t.group_by("registered_number").aggregate(**aggs)

# 3. Safely concatenate the array columns without duplicating tags (e.g., avoiding "01,01")
for col in concat_cols:
    unique_str_agg = (
        t.select("registered_number", col)
        .filter(t[col] != "")
        .distinct()  # Drops duplicates before concatenating
        .group_by("registered_number")
        .aggregate(**{col: ibis._[col].group_concat(",")})
    )
    # Join it back onto our clean base table
    clean_table = clean_table.left_join(unique_str_agg, "registered_number").drop("registered_number_right")

# 4. Execute the replacement safely
con.create_table("fame_derived_clean", clean_table, overwrite=True)
con.create_table("fame_derived", con.table("fame_derived_clean"), overwrite=True)
con.drop_table("fame_derived_clean")

print(f"✅ Deduplicated fame_derived dynamically removing {t_count - clean_table.count().execute()} rows")

✅ Deduplicated fame_derived dynamically removing 49378 rows


In [ ]:
import numpy as np

# Dump the first 500 rows of df_raw, fame_derived, and famed_fixed to a single .xlsx file in /tmp
# Using those as different sheet names
# Get fame_derived and fame_fixed from their ibis tables
out_file_raw = dirs.root_dir / "build" / "tmp" / "df_raw_head.xlsx"
df_raw_head = df_raw.head(500)
df_derived_head = con.table("fame_derived").execute().head(500)
df_fixed_head = con.table("fame_fixed").execute().head(500)
with pd.ExcelWriter(out_file_raw, engine='openpyxl') as writer:
    df_raw_head.to_excel(writer, sheet_name='df_raw', index=False)
    df_derived_head.to_excel(writer, sheet_name='fame_derived', index=False)
    df_fixed_head.to_excel(writer, sheet_name='fame_fixed', index=False)
print(f"✅ Successfully dumped the first 500 rows of df_raw to: {out_file_raw}")


⚠️ Warning: Duplicate registered_number values found in fame_fixed:

Instances of registered_number '06665248' in fame_fixed:
                        company_name registered_number ticker_symbol  \
40452  CALVADNACK LAND TRUST LIMITED          06665248           NaN   
44512  CALVADNACK LAND TRUST LIMITED          06665248           NaN   

      primary_trading_address primary_trading_address_latitude  \
40452                     NaN                              NaN   
44512                     NaN                              NaN   

      primary_trading_address_longitude                    branch_name  \
40452                               NaN  CALVADNACK LAND TRUST LIMITED   
44512                               NaN  CALVADNACK LAND TRUST LIMITED   

       primary_uk_sic_2007_code  \
40452                    1130.0   
44512                    1130.0   

                         primary_uk_sic_2007_description latest_accounts_date  \
40452  Growing of vegetables and melons, roots a

# 4. Geospatial processing

In [ ]:
import ibis
from f_3_spatial import convert_dms_to_decimal
from f_0_dirs import get_data_dirs

dirs = get_data_dirs()
db_path = dirs.output_dir / "fame_data.duckdb"
# con.raw_sql("INSTALL spatial; LOAD spatial;")

# DERIVE
table_fame_fixed: ibis.expr.types.Table   = con.table("fame_fixed")
table_fame_derived: ibis.expr.types.Table = con.table("fame_derived")
table_joined = table_fame_fixed.left_join(table_fame_derived, "registered_number")

# Assume you load a free UK Postcode to Lat/Lon lookup CSV into DuckDB
# table_postcode_lookup = con.table("uk_postcodes") 

# 2. Mutate Hierarchy & Coords
table_mutated = table_joined.mutate(
    
    # --- ADDRESS HIERARCHY ---
    # Returns the first option that isn't Null
    best_full_address = ibis.coalesce(
        table_fame_fixed.primary_trading_address,
        table_fame_fixed.ro_address,
        # Fallback: concatenate the separate lines if the above are null
        ibis.literal(", ").join(
            ibis.array([
                table_fame_fixed.ro_address_line_1, 
                table_fame_fixed.ro_address_line_2, 
                table_fame_fixed.ro_address_postcode
            ]).filter(lambda x: x.notnull()) # Only join non-null lines
        )
    ),
    
    # --- GEOSPATIAL HIERARCHY ---
    # 1. Parse FAME's DMS strings into pure decimal floats
    fame_lat_dec = convert_dms_to_decimal(table_fame_fixed.primary_trading_address_latitude),
    fame_lon_dec = convert_dms_to_decimal(table_fame_fixed.primary_trading_address_longitude),
    
    # 2. (Optional Future Step) If you joined a postcode lookup table, 
    # you would include its lat/lon here as a fallback
    # lookup_lat = table_postcode_lookup.latitude,
    
    # 3. Store the best available coordinates
    best_latitude = ibis.coalesce(
        convert_dms_to_decimal(table_fame_fixed.primary_trading_address_latitude),
        # lookup_lat
    )
)

# 3. Select Derived Schema Columns and Save
table_derived = table_mutated.select(schema_derived_names)
con.create_table("fame_derived", table_derived, overwrite=True)